# PRUEBAS

In [0]:
"""
import pandas as pd, tempfile, os

one = files[0]
tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".xls").name
dbutils.fs.cp(one, f"file:{tmp}")

# prueba nombres de índice
indice_name = None
for n in ["Índice", "INDICE", "ÍNDICE", "Indice", "indice", "índice"]:
    try:
        ind = pd.read_excel(tmp, sheet_name=n, header=None, engine="xlrd")
        indice_name = n
        break
    except:
        pass

print("Índice sheet:", indice_name)
print("shape:", ind.shape)
print("primeras 5 filas, primeras 15 columnas:")
display(ind.iloc[:5, :15])

# mira si hay números 1..60 en alguna columna
nums = {}
for c in range(min(25, ind.shape[1])):
    col = ind.iloc[:, c]
    # cuenta cuántos valores parecen números 1..60
    cnt = 0
    for v in col.dropna().head(200).tolist():
        s = str(v).strip()
        if s.isdigit():
            k = int(s)
            if 1 <= k <= 60:
                cnt += 1
    nums[c] = cnt

print("conteo de posibles números por columna (top):", sorted(nums.items(), key=lambda x: -x[1])[:10])

os.remove(tmp)
"""

In [0]:
"""import re
import pandas as pd
import os
import tempfile
from pyspark.sql.functions import lit

_indice_cache = {}

def _download_to_tmp(abfss_path: str, suffix: str = ".xls") -> str:
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp_path = tmp.name
    tmp.close()
    dbutils.fs.cp(abfss_path, f"file:{tmp_path}")
    return tmp_path

def _build_sheet_map_from_indice(local_xls_path: str) -> dict:
    ind = None
    for indice_name in ["Índice", "INDICE", "ÍNDICE", "Indice", "indice", "índice"]:
        try:
            ind = pd.read_excel(local_xls_path, sheet_name=indice_name, header=None, engine="xlrd")
            break
        except:
            pass
    if ind is None:
        raise Exception("No pude leer la hoja Índice con pandas/xlrd (nombre no encontrado).")

    sheet_map = {}

    for _, row in ind.iterrows():
        values = row.tolist()

        num = None
        for v in values:
            if pd.isna(v): 
                continue
            try:
                k = int(float(v))
                if 1 <= k <= 60:
                    num = k
                    break
            except (ValueError, TypeError):
                continue

        if num is None:
            continue

        text_candidates = []
        for v in values:
            if pd.isna(v):
                continue
            sv = str(v).strip()
            if len(sv) >= 6:
                try:
                    float(sv)
                except ValueError:
                    text_candidates.append(sv)

        if not text_candidates:
            continue

        title = max(text_candidates, key=len)
        sheet_map[title] = num

    if len(sheet_map) == 0:
        raise Exception("Leí la hoja Índice pero no pude construir el mapeo (título -> número).")

    return sheet_map

def get_sheet_number_from_title(path_abfss: str, target_title: str) -> int:
    if path_abfss in _indice_cache:
        sheet_map = _indice_cache[path_abfss]
    else:
        local_path = _download_to_tmp(path_abfss, suffix=".xls")
        try:
            sheet_map = _build_sheet_map_from_indice(local_path)
            _indice_cache[path_abfss] = sheet_map
        finally:
            if os.path.exists(local_path):
                os.remove(local_path)

    tgt = target_title.lower()
    for title, num in sheet_map.items():
        if tgt in title.lower():
            return num

    suggestions = [t for t in sheet_map.keys() if any(w in t.lower() for w in tgt.split()[:2])]
    raise Exception(f"No encontré '{target_title}' en índice. Ejemplos cercanos: {suggestions[:8]}")

def read_xls_sheet_by_number(path_abfss: str, sheet_number: int):
    local_path = _download_to_tmp(path_abfss, suffix=".xls")
    try:
        pdf = pd.read_excel(local_path, sheet_name=sheet_number, header=None, engine="xlrd")
        pdf = pdf.astype(str)
        return spark.createDataFrame(pdf)
    finally:
        if os.path.exists(local_path):
            os.remove(local_path)
"""

In [0]:
"""# Hojas objetivo por “título” tal como aparece en el índice (usa contains)
TARGET_TITLES = [
    "Ratios de Morosidad según días de incumplimiento",
    "Morosidad por tipo de crédito y modalidad"
]

all_dfs = []

for file_path in files:
    file_name = file_path.split("/")[-1]

    m = re.search(r"-([a-z]{2})\d{4}", file_name.lower())
    periodo = m.group(1) if m else "unknown"

    for title in TARGET_TITLES:
        try:
            sheet_num = get_sheet_number_from_title(file_path, title)
            df = read_xls_sheet_by_number(file_path, sheet_num)

            df = (df.withColumn("source_file", lit(file_name))
                    .withColumn("periodo_mes", lit(periodo))
                    .withColumn("sheet_title", lit(title))
                    .withColumn("sheet_num", lit(int(sheet_num))))

            all_dfs.append(df)

        except Exception as e:
            # En algunos meses puede variar el título o faltar la tabla.
            print(f"[WARN] {file_name} - no pude leer '{title}': {e}")

if len(all_dfs) == 0:
    raise Exception("No se pudo leer ninguna hoja objetivo en ningún archivo.")

df_final = all_dfs[0]
for d in all_dfs[1:]:
    df_final = df_final.unionByName(d, allowMissingColumns=True)

display(df_final.limit(30))
print("Total filas:", df_final.count())
"""

In [0]:
"""import pandas as pd
import re, os, tempfile
from pyspark.sql.functions import lit

def _download_to_tmp(abfss_path: str, suffix=".xls") -> str:
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp_path = tmp.name
    tmp.close()
    dbutils.fs.cp(abfss_path, f"file:{tmp_path}")
    return tmp_path

def _find_sheet_num_in_row(row_vals):
    # Busca un entero 1..60 dentro de la fila
    for v in row_vals:
        if pd.isna(v): 
            continue
        try:
            k = int(float(v))
            if 1 <= k <= 60:
                return k
        except:
            pass
    return None

def get_sheet_number_from_index_row(local_xls_path: str, target_title: str) -> int:
    # Lee hoja índice (probar variantes)
    ind = None
    for indice_name in ["Índice","INDICE","ÍNDICE","Indice","indice","índice"]:
        try:
            ind = pd.read_excel(local_xls_path, sheet_name=indice_name, header=None, engine="xlrd")
            break
        except:
            pass
    if ind is None:
        raise Exception("No pude leer hoja Índice")

    tgt = target_title.lower().strip()

    for _, row in ind.iterrows():
        vals = ["" if pd.isna(v) else str(v).strip() for v in row.tolist()]
        row_text = " | ".join(vals).lower()
        if tgt in row_text:
            num = _find_sheet_num_in_row(row.tolist())
            if num:
                return num

    # fallback: match por palabras (por si el índice acorta el texto)
    words = [w for w in re.split(r"\s+", tgt) if len(w) > 4][:3]
    for _, row in ind.iterrows():
        vals = ["" if pd.isna(v) else str(v).strip() for v in row.tolist()]
        row_text = " | ".join(vals).lower()
        if all(w in row_text for w in words):
            num = _find_sheet_num_in_row(row.tolist())
            if num:
                return num

    raise Exception(f"No hallé el sheet_num para: {target_title}")"""

In [0]:
""" test_file = files[0]
local_path = _download_to_tmp(test_file, suffix=".xls")

import xlrd
wb = xlrd.open_workbook(local_path)
print("Primeras hojas:", wb.sheet_names()[:10])
print("Existe hoja '13'?:", "13" in wb.sheet_names())
print("Existe hoja '16'?:", "16" in wb.sheet_names())

os.remove(local_path) """

In [0]:
""" 
spark.sql(f"""
""" SELECT sheet_num, sheet_title, COUNT(*) 
FROM {catalog}.bronze.bronze_sbs_morosidad_2024
GROUP BY sheet_num, sheet_title
""")""".show() """

In [0]:
"""df_final.filter("sheet_num = 13").limit(30).display()
df_final.filter("sheet_num = 16").limit(30).display()
"""

# INICIANDO

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("storageName", "adlsbsriskdev")
dbutils.widgets.text("catalogName", "sbsrisk_dev")
dbutils.widgets.text("year", "2025")   # cambiar el año de procesamiento

storage = dbutils.widgets.get("storageName")
catalog = dbutils.widgets.get("catalogName")
year = dbutils.widgets.get("year")

spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA bronze")

In [0]:
raw_base = f"abfss://raw@{storage}.dfs.core.windows.net/sbs/{year}/"
display(dbutils.fs.ls(raw_base))

In [0]:
files = [f.path for f in dbutils.fs.ls(raw_base) 
         if f.path.lower().endswith(".xls") or f.path.lower().endswith(".xlsx")]

print("Total excels encontrados:", len(files))
for f in files[:5]:
    print(f)

if len(files) == 0:
    raise Exception(f"No se encontraron archivos .XLS/.XLSX en: {raw_base}")

In [0]:
# Dependencias para leer xls
%pip install xlrd==2.0.1

In [0]:
import pandas as pd
import re, os, tempfile
from pyspark.sql import functions as F

# ========= Helpers =========
def _download_to_tmp(abfss_path: str, suffix=".xls") -> str:
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    tmp_path = tmp.name
    tmp.close()
    dbutils.fs.cp(abfss_path, f"file:{tmp_path}")
    return tmp_path

def _find_sheet_num_in_row(row_vals):
    nums = []
    for v in row_vals:
        if pd.isna(v):
            continue
        try:
            k = int(float(v))
            if 1 <= k <= 60:
                nums.append(k)
        except:
            pass
    # en estos índices el número correcto suele estar al final
    return nums[-1] if nums else None

def get_sheet_number_from_index_row(local_xls_path: str, target_title: str) -> int:
    ind = None
    for indice_name in ["Índice","INDICE","ÍNDICE","Indice","indice","índice"]:
        try:
            ind = pd.read_excel(local_xls_path, sheet_name=indice_name, header=None, engine="xlrd")
            break
        except:
            pass
    if ind is None:
        raise Exception("No pude leer hoja Índice")

    tgt = target_title.lower().strip()

    # 1) Match exacto por fila (contains)
    for _, row in ind.iterrows():
        vals = ["" if pd.isna(v) else str(v).strip() for v in row.tolist()]
        row_text = " | ".join(vals).lower()
        if tgt in row_text:
            num = _find_sheet_num_in_row(row.tolist())
            if num:
                return num

    # 2) Fallback: match por 2-3 palabras largas
    words = [w for w in re.split(r"\s+", tgt) if len(w) > 4][:3]
    for _, row in ind.iterrows():
        vals = ["" if pd.isna(v) else str(v).strip() for v in row.tolist()]
        row_text = " | ".join(vals).lower()
        if all(w in row_text for w in words):
            num = _find_sheet_num_in_row(row.tolist())
            if num:
                return num

    raise Exception(f"No hallé sheet_num para: {target_title}")

def read_xls_sheet_by_number(path_abfss: str, sheet_number: int):
    local_path = _download_to_tmp(path_abfss, suffix=".xls")
    try:
        # IMPORTANTE: leer por NOMBRE de hoja ("13"), no por índice (13)
        pdf = pd.read_excel(local_path, sheet_name=str(sheet_number), header=None, engine="xlrd")
        pdf = pdf.astype(str)
        return spark.createDataFrame(pdf)
    finally:
        if os.path.exists(local_path):
            os.remove(local_path)

def sheet_looks_like_morosidad(df_spark):
    # revisa primeras filas buscando palabras clave
    sample = df_spark.limit(30)
    cols = sample.columns[:12]  # primeras 12 cols suelen bastar
    txt = (sample
           .select(F.lower(F.concat_ws(" | ", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols])).alias("t"))
           .groupBy()
           .agg(F.concat_ws(" || ", F.collect_list("t")).alias("alltxt"))
           .collect()[0]["alltxt"])
    return ("moros" in txt) or ("días" in txt) or ("dias" in txt)

In [0]:
from pyspark.sql.functions import lit

# Títulos tal como aparecen en el índice (o muy similares)
TARGET_TITLES = [
    "Ratios de Morosidad según días de incumplimiento",
    "Morosidad por tipo de crédito y modalidad"
]

all_dfs = []

for file_path in files:
    file_name = file_path.split("/")[-1]
    m = re.search(r"-([a-z]{2})\d{4}", file_name.lower())
    periodo = m.group(1) if m else "unknown"

    # Descarga local solo para leer el índice UNA VEZ por archivo
    local_xls = _download_to_tmp(file_path, suffix=".xls")
    try:
        for title in TARGET_TITLES:
            try:
                sheet_num = get_sheet_number_from_index_row(local_xls, title)
                df = read_xls_sheet_by_number(file_path, sheet_num)
                
                # Validación: si no parece morosidad, probamos con el "penúltimo" número de la fila
                if not sheet_looks_like_morosidad(df) and "moros" in title.lower():
                    # re-leer índice y extraer todos los nums para intentar otra opción
                    ind = pd.read_excel(local_xls, sheet_name="Índice", header=None, engine="xlrd")
                    tgt = title.lower().strip()

                    candidate_nums = []
                    for _, row in ind.iterrows():
                        vals = ["" if pd.isna(v) else str(v).strip() for v in row.tolist()]
                        row_text = " | ".join(vals).lower()
                        if tgt in row_text:
                            # saca TODOS los ints 1..60 de esa fila
                            nums = []
                            for v in row.tolist():
                                try:
                                    k = int(float(v))
                                    if 1 <= k <= 60:
                                        nums.append(k)
                                except:
                                    pass
                            candidate_nums = nums

                    if len(candidate_nums) >= 2:
                        alt_num = candidate_nums[-2]  # penúltimo
                        df_alt = read_xls_sheet_by_number(file_path, alt_num)
                        if sheet_looks_like_morosidad(df_alt):
                            sheet_num = alt_num
                            df = df_alt

                df = (df.withColumn("source_file", lit(file_name))
                        .withColumn("periodo_mes", lit(periodo))
                        .withColumn("sheet_title", lit(title))
                        .withColumn("sheet_num", lit(int(sheet_num))))

                all_dfs.append(df)
            except Exception as e:
                print(f"[WARN] {file_name} - {title}: {e}")
    finally:
        if os.path.exists(local_xls):
            os.remove(local_xls)

if len(all_dfs) == 0:
    raise Exception("No se pudo leer ninguna hoja objetivo en ningún archivo.")

df_final = all_dfs[0]
for d in all_dfs[1:]:
    df_final = df_final.unionByName(d, allowMissingColumns=True)

display(df_final.limit(30))
print("Total filas:", df_final.count())

In [0]:
from pyspark.sql import functions as F

# Quita filas donde TODAS las columnas de datos sean "nan" / "None" / vacío
data_cols = [c for c in df_final.columns if not c in ["source_file","periodo_mes","sheet_title","sheet_num"]]

df_bronze = df_final
for c in data_cols:
    df_bronze = df_bronze.withColumn(c, F.when(F.col(c).isin("nan","None",""), None).otherwise(F.col(c)))

df_bronze = df_bronze.na.drop("all", subset=data_cols)

display(df_bronze.limit(30))
print("Filas luego de limpiar vacíos:", df_bronze.count())

In [0]:
bronze_path = f"abfss://bronze@{storage}.dfs.core.windows.net/{catalog}/bronze/bronze_sbs_morosidad_{year}"

(df_bronze.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(bronze_path))

print("OK escrito en:", bronze_path)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.bronze.bronze_sbs_morosidad_{year}
USING DELTA
LOCATION '{bronze_path}'
""")

spark.sql(f"SELECT COUNT(*) AS total_rows FROM {catalog}.bronze.bronze_sbs_morosidad_{year}").show()

In [0]:
display(dbutils.fs.ls(bronze_path))